In [ ]:
import requests
import pandas as pd

In [ ]:
complete = False
page = 1

In [ ]:
dict = []

while not complete:
  data = requests.get(f"https://collections.leventhalmap.org/search.json?per_page=100&search_field=all_fields&q=&search_field=all_fields&page={page}")
  thisPageResult = data.json()
  allPages = thisPageResult['response']['pages']['total_pages']
  print(f'💬 processing page {page}/{allPages}')

  for record in thisPageResult['response']['docs']:
    
    manifest = f'https://www.digitalcommonwealth.org/search/{record["id"]}/manifest'
    iiifResponse = requests.get(manifest)
    
    try:
      
      w = iiifResponse.json()['sequences'][0]['canvases'][0]['width']
      h = iiifResponse.json()['sequences'][0]['canvases'][0]['height']
      ar = w/h

      if ar > 3:

        dict.append(
          {
            'id': record["id"],
            'w': w,
            'h': h,
            'ar': ar
          }
        )

    except:
      pass
    
    complete = True if thisPageResult['response']['pages']['last_page?'] else False
    page = thisPageResult['response']['pages']['next_page']
  df = pd.DataFrame(dict)
  df.to_csv('out.csv', index=False)
  print(f"💾 saved through page {page}/{allPages}!")